# AML/BSA Compliance Gaps in the Brazil–U.S. Financial Corridor
## Quantifying Regulatory Exposure for SEC-Registered Investment Advisers

**Author:** Wederson Marinho dos Santos  
**Affiliation:** Kadima Holding | ORCID: 0009-0004-6401-3465  
**Date:** March 2026  
**Companion paper:** *LGPD Compliance for U.S. Investment Advisers with Brazilian Client Exposure* (SSRN, 2026)

---

### Abstract

This notebook quantifies the regulatory exposure created by the intersection of the Bank Secrecy Act (BSA) and Brazil's Lei Geral de Proteção de Dados (LGPD) for SEC-registered investment advisers with Brazilian client exposure. Using public data from FinCEN Suspicious Activity Report (SAR) statistics (2014–2024) and SEC Form ADV aggregate data (2017–2023), we document: (1) a 1,009% growth in investment adviser SAR filings between 2014 and 2023, establishing the sector's materiality as an AML actor; (2) a 288% surge in Florida-based securities SARs between 2019 and 2024, consistent with the concentration of Brazilian wealth management activity in that state; (3) $30.3 trillion in non-US RAUM managed by RIAs as of December 2023, representing 23.5% of total RIA assets and constituting the population of potentially LGPD-exposed data processing. A regulatory gap scoring model identifies advisers most likely to face simultaneous BSA compliance obligations and LGPD data minimization conflicts. Findings extend the analytical framework developed in the companion paper by providing empirical scale to the legal conflicts identified therein.

**Data sources:** FinCEN SAR Statistics (public, fincen.gov); SEC Investment Adviser Statistics (public, sec.gov)  
**JEL codes:** G28, K22, C55, F38  
**Keywords:** AML/BSA, LGPD, investment advisers, cross-border compliance, FinCEN, data protection, Brazil-USA

---


## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Matplotlib style: clean, publication-ready ──────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linewidth': 0.8,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

NAVY  = '#0A2342'
GOLD  = '#C9A55A'
TEAL  = '#2CA58D'
RED   = '#C1440E'
GRAY  = '#6B7280'

print("Libraries loaded. Ready to build datasets.")
print(f"pandas {pd.__version__} | numpy {np.__version__}")


## 2. Dataset 1 — FinCEN SAR Statistics (2014–2024)

**Source:** FinCEN SAR Filing Trend Data, publicly available at  
`https://www.fincen.gov/reports/sar-stats/sar-filings-industry`

Three sectors downloaded:
- **Securities/Futures** (Section 5) — covers investment advisers
- **Money Services Business** (Section 4) — covers remittance channels
- **Depository Institutions** (Section 2) — covers correspondent banking

All data reflects FinCEN Form 111 (BSA SAR) filings, period January 1, 2014 – December 31, 2024.


In [ ]:
# ── Investment Adviser SAR series (from Securities/Futures Exhibit 9) ──────
# Source: Section 5 - Securities and Futures SARs.xlsx, Exhibit 9
# Verified from raw data extraction — see data_collection_notes.md

ia_sars = pd.DataFrame({
    'year': list(range(2014, 2025)),
    'investment_adviser_sars': [
        1638, 2285, 2104, 4091, 5910,
        8423, 11087, 13546, 17219, 18170, 11935
    ]
})

# ── Annual totals by sector ──────────────────────────────────────────────────
sector_totals = pd.DataFrame({
    'year': list(range(2014, 2025)),
    'securities_futures_total': [
        22077, 20379, 18938, 23832, 26776,
        33222, 38236, 58951, 61783, 61029, 61533
    ],
    'msb_total': [
        706163, 799073, 870220, 874130, 873477,
        852925, 917779, 1137454, 1195577, 1266826, 1161768
    ],
    'depository_total': [
        839314, 879907, 958537, 916283, 977703,
        1116400, 1211345, 1426741, 1827918, 1963057, 2037214
    ]
})

# ── Florida Securities SARs (Brazil-facing geographic proxy) ─────────────────
# Source: Section 5 - Securities and Futures SARs.xlsx, Exhibit 2
fl_sars = pd.DataFrame({
    'year': list(range(2014, 2025)),
    'florida_securities_sars': [
        1303, 1318, 1504, 1621, 1511,
        5546, 4782, 18146, 14638, 20384, 21510
    ]
})

# ── MSB Suspicious Wire/EFT Transfers (remittance channel proxy) ─────────────
# Source: Section 4 - Money Services Business SARs.xlsx, Exhibit 5
msb_wire = pd.DataFrame({
    'year': list(range(2015, 2025)),
    'msb_suspicious_eft_wire': [
        45312, 55063, 50499, 44099, 47599,
        74879, 86734, 89348, 94345, None
    ]
})
# 2024 wire data not separately available in public aggregate; set to NaN
msb_wire = msb_wire.dropna()

print("Datasets loaded:")
print(f"  IA SARs: {len(ia_sars)} years (2014–2024)")
print(f"  Sector totals: {len(sector_totals)} years")
print(f"  FL Securities SARs: {len(fl_sars)} years")
print(f"  MSB wire/EFT: {len(msb_wire)} years")
print()
print(ia_sars.to_string(index=False))


## 3. Dataset 2 — SEC Investment Adviser Statistics (Form ADV, 2017–2023)

**Source:** SEC Division of Investment Management Analytics Office,  
*Investment Adviser Statistics: Form ADV Data, period ending December 2023* (May 15, 2024)  
`https://www.sec.gov/files/im-investment-adviser-statistics-20240515.pdf`

Key tables used:
- **Table 1.1** — Total RIAs and ERAs by year
- **Table 2.4** — US vs Non-US RAUM (proxy for foreign client exposure)
- **Table 3.3** — Number of RIAs by client type (Foreign Institutions)
- **Table 7.2** — RIAs by principal office country


In [ ]:
# ── RIA/ERA universe ────────────────────────────────────────────────────────
ria_universe = pd.DataFrame({
    'year': list(range(2009, 2024)),
    'total_rias': [
        11458, 11719, 10733, 10662, 10826, 11396, 11855,
        12065, 12572, 13101, 13349, 13783, 14699, 15287, 15441
    ],
    'total_eras': [
        0, 1, 2256, 2560, 2781, 3072, 3417,
        3602, 3950, 4266, 4624, 5095, 5646, 6046, 5762
    ]
})
ria_universe['total_entities'] = ria_universe['total_rias'] + ria_universe['total_eras']

# ── Non-US RAUM (Table 2.4) ──────────────────────────────────────────────────
non_us_raum = pd.DataFrame({
    'year': [2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'us_raum_trillions':     [65.0, 64.0, 73.7, 82.1, 94.7, 87.1, 98.6],
    'non_us_raum_trillions': [18.6, 22.4, 25.3, 28.0, 33.4, 28.3, 30.3],
    'total_raum_trillions':  [83.6, 86.4, 99.1, 110.2, 128.1, 115.4, 128.8]
})
non_us_raum['non_us_pct'] = (
    non_us_raum['non_us_raum_trillions'] / non_us_raum['total_raum_trillions'] * 100
)

# ── RIAs with Foreign Institution clients (Table 3.3) ────────────────────────
foreign_inst_rias = pd.DataFrame({
    'year': [2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'rias_with_foreign_institution_clients': [908, 928, 855, 787, 746, 722, 695]
})

print("SEC Form ADV datasets loaded:")
print(f"  RIA universe: 2009–2023")
print(f"  Non-US RAUM: 2017–2023")
print(f"  Foreign institution RIAs: 2017–2023")
print()
print("Non-US RAUM summary:")
print(non_us_raum[['year','non_us_raum_trillions','non_us_pct']].to_string(index=False))


## 4. Analysis and Visualizations

### Figure 1 — Investment Adviser SAR Filings (2014–2024)

The 1,009% growth in investment adviser SARs between 2014 and 2023 establishes the sector's materiality as an AML actor — and makes the FinCEN rule's extension of BSA obligations to investment advisers analytically predictable. The 34.3% contraction in 2024 coincides with FinCEN's August 2025 exemptive relief order and the formal delay of the rule's compliance deadline to January 1, 2028.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Left: IA SAR absolute series ─────────────────────────────────────────────
ax1 = axes[0]
bars = ax1.bar(ia_sars['year'], ia_sars['investment_adviser_sars'],
               color=[RED if y == 2024 else NAVY for y in ia_sars['year']],
               edgecolor='white', linewidth=0.5, alpha=0.92)

# Annotate 2024 bar
ax1.annotate('–34.3%\n(Rule delay)', xy=(2024, 11935),
             xytext=(2022.2, 14500),
             fontsize=9, color=RED, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

ax1.set_title('Investment Adviser SAR Filings
2014–2024', pad=10)
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of SAR Filings')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_xticks(ia_sars['year'])
ax1.tick_params(axis='x', rotation=45)

# ── Right: IA share of total Securities SARs ─────────────────────────────────
ax2 = axes[1]
merged = ia_sars.merge(sector_totals[['year','securities_futures_total']], on='year')
merged['ia_share_pct'] = merged['investment_adviser_sars'] / merged['securities_futures_total'] * 100

ax2.plot(merged['year'], merged['ia_share_pct'],
         color=TEAL, linewidth=2.5, marker='o', markersize=7, label='IA share')
ax2.fill_between(merged['year'], merged['ia_share_pct'], alpha=0.15, color=TEAL)
ax2.axvline(x=2024, color=RED, linestyle='--', linewidth=1.2, alpha=0.7,
            label='FinCEN Rule delay (2024)')

ax2.set_title('Investment Advisers as Share of\nAll Securities/Futures SARs', pad=10)
ax2.set_xlabel('Year')
ax2.set_ylabel('Percentage (%)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax2.set_xticks(merged['year'])
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

# ── Shared footer ─────────────────────────────────────────────────────────────
fig.text(0.5, -0.02,
    'Source: FinCEN SAR Filing Trend Data, Section 5 — Securities and Futures (2014–2024). '
    'fincen.gov/reports/sar-stats',
    ha='center', fontsize=8.5, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('figure1_ia_sar_series.png', bbox_inches='tight', dpi=150)
plt.show()
print("Figure 1 saved.")

# Print key statistics
print(f"\nKey statistics:")
print(f"  IA SARs 2014: {1638:,}")
print(f"  IA SARs 2023 (peak): {18170:,}")
print(f"  Growth 2014–2023: +{((18170-1638)/1638*100):.0f}%")
print(f"  IA SARs 2024: {11935:,}")
print(f"  Drop 2023–2024: {((11935-18170)/18170*100):.1f}%")
print(f"  IA share of Securities SARs 2019: {8423/33222*100:.1f}%")
print(f"  IA share of Securities SARs 2023: {18170/61029*100:.1f}%")


### Figure 2 — Florida Securities SARs as Brazil-Facing Geographic Proxy

Florida hosts the largest concentration of Brazilian-born residents in the United States and the most significant cluster of wealth management relationships between U.S. advisers and Brazilian clients. The 288% surge in Florida securities SARs between 2019 and 2024 is consistent with — though not exclusively attributable to — the growth of cross-border financial activity in this corridor. The 227% spike in 2021 coincides with record-high Brazil-to-U.S. capital outflows documented by BACEN.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Left: Florida vs NY vs TX ─────────────────────────────────────────────────
ax1 = axes[0]
ny_sars = [2763, 3047, 2933, 3529, 3880, 5589, 6457, 7987, 10902, 14280, 13779]
tx_sars = [836, 982, 1147, 1267, 1250, 1369, 954, 740, 891, 880, 11035]
fl_vals  = [1303, 1318, 1504, 1621, 1511, 5546, 4782, 18146, 14638, 20384, 21510]
years    = list(range(2014, 2025))

ax1.plot(years, fl_vals, color=GOLD,  linewidth=2.8, marker='o', markersize=7, label='Florida')
ax1.plot(years, ny_sars, color=NAVY,  linewidth=2.0, marker='s', markersize=6, label='New York')
ax1.plot(years, tx_sars, color=TEAL,  linewidth=2.0, marker='^', markersize=6, label='Texas')

ax1.annotate('FL spike\n+227% in 2021', xy=(2021, 18146),
             xytext=(2018.5, 17000),
             fontsize=9, color=GOLD, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=GOLD, lw=1.5))

ax1.set_title('Securities SARs: FL vs NY vs TX
2014–2024', pad=10)
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of SAR Filings')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_xticks(years)
ax1.tick_params(axis='x', rotation=45)
ax1.legend()

# ── Right: Florida growth index (2019=100) ────────────────────────────────────
ax2 = axes[1]
fl_idx = [v / 5546 * 100 for v in fl_vals]
ny_idx = [v / 5589 * 100 for v in ny_sars]
tx_idx = [v / 1369 * 100 for v in tx_sars]

ax2.plot(years, fl_idx, color=GOLD,  linewidth=2.8, marker='o', markersize=7, label='Florida')
ax2.plot(years, ny_idx, color=NAVY,  linewidth=2.0, marker='s', markersize=6, label='New York')
ax2.plot(years, tx_idx, color=TEAL,  linewidth=2.0, marker='^', markersize=6, label='Texas')
ax2.axhline(y=100, color=GRAY, linestyle='--', linewidth=1.0, alpha=0.6)
ax2.text(2014.1, 104, '2019 = 100', fontsize=8.5, color=GRAY, style='italic')

ax2.set_title('Securities SARs Growth Index\n(2019 = 100)', pad=10)
ax2.set_xlabel('Year')
ax2.set_ylabel('Index (2019 = 100)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
ax2.set_xticks(years)
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

fig.text(0.5, -0.02,
    'Source: FinCEN SAR Filing Trend Data, Section 5 — Securities and Futures, Exhibit 2 (2014–2024). '
    'fincen.gov/reports/sar-stats',
    ha='center', fontsize=8.5, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('figure2_florida_geographic_proxy.png', bbox_inches='tight', dpi=150)
plt.show()
print("Figure 2 saved.")
print(f"\nFlorida securities SARs growth 2019–2024: +{((21510-5546)/5546*100):.0f}%")
print(f"New York securities SARs growth 2019–2024: +{((13779-5589)/5589*100):.0f}%")
print(f"Texas securities SARs growth 2019–2024: +{((11035-1369)/1369*100):.0f}%")


### Figure 3 — Non-US RAUM and Foreign Client Exposure (SEC Form ADV, 2017–2023)

As of December 2023, RIAs managed $30.3 trillion in non-US client assets, representing 23.5% of total RIA RAUM. Each dollar of non-US RAUM represents personal data of non-U.S. persons processed by a U.S.-regulated entity — the precise scenario in which the BSA-LGPD conflict analyzed in the companion paper operates. The 695 RIAs reporting Foreign Institution clients as of 2023 represent the minimum observable population of advisers with documented cross-border institutional client relationships.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Left: Non-US RAUM absolute ────────────────────────────────────────────────
ax1 = axes[0]
bars = ax1.bar(non_us_raum['year'],
               non_us_raum['non_us_raum_trillions'],
               color=NAVY, edgecolor='white', linewidth=0.5, alpha=0.88)

ax1.plot(non_us_raum['year'], non_us_raum['non_us_raum_trillions'],
         color=GOLD, linewidth=2.0, marker='D', markersize=7, label='Non-US RAUM trend')

ax1.set_title('Non-US RAUM Managed by RIAs\n($Trillions, 2017–2023)', pad=10)
ax1.set_xlabel('Year')
ax1.set_ylabel('Trillions USD')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}T'))
ax1.set_xticks(non_us_raum['year'])
ax1.legend()

# ── Right: Foreign Institution RIAs + Non-US % dual axis ─────────────────────
ax2 = axes[1]
ax2b = ax2.twinx()

ax2.bar(foreign_inst_rias['year'],
        foreign_inst_rias['rias_with_foreign_institution_clients'],
        color=TEAL, alpha=0.75, label='RIAs w/ Foreign Institution clients')

ax2b.plot(non_us_raum['year'], non_us_raum['non_us_pct'],
          color=RED, linewidth=2.5, marker='o', markersize=7,
          label='Non-US RAUM %')
ax2b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

ax2.set_title('Foreign Exposure: RIAs with\nForeign Institution Clients (2017–2023)', pad=10)
ax2.set_xlabel('Year')
ax2.set_ylabel('Number of RIAs')
ax2b.set_ylabel('Non-US RAUM as % of Total')

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

fig.text(0.5, -0.02,
    'Source: SEC Investment Adviser Statistics, Form ADV Data, period ending December 2023 '
    '(Tables 2.4, 3.3). sec.gov',
    ha='center', fontsize=8.5, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('figure3_sec_foreign_exposure.png', bbox_inches='tight', dpi=150)
plt.show()
print("Figure 3 saved.")
print(f"\nNon-US RAUM 2023: ${30.3}T ({23.5}% of total)")
print(f"Peak Non-US RAUM: 2021 at ${33.4}T ({26.1}%)")
print(f"RIAs with Foreign Institution clients 2018 (peak): 928")
print(f"RIAs with Foreign Institution clients 2023: 695 (–25.1% since peak)")


### Figure 4 — Regulatory Timeline: The Compliance Urgency Window (2018–2028)

The convergence of regulatory events between 2018 and 2028 defines the compliance urgency window analyzed in the companion paper. LGPD enacted in 2018 and enforced since 2021; FinCEN rule published September 2024, delayed to January 2028; ANPD SCCs mandatory since August 2025. Each event tightens the compliance perimeter for advisers with Brazilian client exposure.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(2017.5, 2029)
ax.set_ylim(-0.5, 5.5)
ax.set_yticks([])
ax.set_xticks(range(2018, 2029))
ax.tick_params(axis='x', labelsize=10)
ax.set_xlabel('Year', fontsize=11)
ax.set_title('Regulatory Timeline: BSA-LGPD Compliance Convergence (2018–2028)',
             fontsize=13, fontweight='bold', pad=14)
ax.axhline(y=0, color='#cccccc', linewidth=1.0, zorder=0)

events = [
    (2018.6,  5.0, 'LGPD enacted\n(Law 13,709/2018)', NAVY,   'Brazil'),
    (2020.7,  4.0, 'LGPD enforcement\nstarts (Sept 2020)', NAVY,   'Brazil'),
    (2021.8,  3.0, 'ANPD administrative\npenalties active', NAVY,   'Brazil'),
    (2024.7,  5.0, 'FinCEN IA Rule\npublished (89 FR 72156)', RED,    'USA'),
    (2024.85, 2.0, 'FinCEN rule\ndelayed to 2028', RED,    'USA'),
    (2025.65, 4.0, 'ANPD SCCs mandatory\n(Resolution 19/2024)', GOLD,   'Both'),
    (2025.75, 1.0, 'ANPD → independent\nregulatory agency\n(MP 1317/2025)', NAVY,   'Brazil'),
    (2026.0,  3.0, 'ANPD EU adequacy\ndecision (Res. 32/2026)', GOLD,   'Both'),
    (2028.0,  5.0, 'FinCEN IA Rule\ncompliance deadline', RED,    'USA'),
]

legend_colors = {'Brazil': NAVY, 'USA': RED, 'Both / Bilateral': GOLD}

for x, y, label, color, jurisdiction in events:
    ax.scatter(x, y, s=120, color=color, zorder=5, edgecolors='white', linewidth=1.5)
    va = 'bottom' if y > 2.5 else 'top'
    offset = 0.28 if va == 'bottom' else -0.28
    ax.annotate(label,
                xy=(x, y), xytext=(x, y + offset),
                ha='center', va=va, fontsize=8.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=color, alpha=0.9, linewidth=1.0))

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=NAVY, markersize=9, label='Brazil (LGPD/ANPD)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=RED,  markersize=9, label='USA (BSA/FinCEN)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=GOLD, markersize=9, label='Bilateral'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9.5,
          framealpha=0.95, edgecolor='#cccccc')

# Urgency window shading
ax.axvspan(2024.6, 2028.1, alpha=0.06, color=RED, label='Compliance urgency window')
ax.text(2026.3, 0.2, 'Compliance urgency window
(FinCEN rule published → deadline)',
        ha='center', fontsize=8.5, color=RED, style='italic', alpha=0.8)

fig.text(0.5, -0.04,
    'Sources: 89 Fed. Reg. 72,156 (FinCEN, Sept. 2024); LGPD Law 13,709/2018; '
    'ANPD Resolution No. 19/2024; ANPD Resolution No. 32/2026; MP 1,317/2025.',
    ha='center', fontsize=8, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('figure4_regulatory_timeline.png', bbox_inches='tight', dpi=150)
plt.show()
print("Figure 4 saved.")


### Figure 5 — Compliance Gap Heatmap: BSA vs LGPD Obligation Conflicts

This heatmap systematizes the four conflict categories identified in Section 4 of the companion paper, mapping each BSA obligation against the specific LGPD provisions it conflicts with. Conflict severity is scored 1 (low) to 3 (high) based on: (1) whether the conflict is directly contradictory or merely creates ambiguity; (2) whether regulatory guidance has addressed it; and (3) the enforcement risk under current ANPD supervisory priorities.


In [ ]:
import matplotlib.colors as mcolors

# ── Conflict matrix: BSA obligations × LGPD provisions ───────────────────────
bsa_obligations = [
    'CIP: Data Collection\n(Art. 7 Legal Basis)',
    'SAR Filing:\nTipping-Off Prohibition',
    'Retention:\n5-Year CIP Records',
    'Cross-Border Transfer:\nSAR to FinCEN',
    'Enhanced Due\nDiligence (EDD)',
]

lgpd_provisions = [
    'Data Minimization\n(Art. 6, III)',
    'Transparency\n(Art. 6, VI)',
    'Retention Limits\n(Art. 16)',
    'Int'l Transfer\nRestrictions (Art. 33)',
    'Legal Obligation\nBasis (Art. 7, II)',
]

# Severity scores: 0=no conflict, 1=ambiguity, 2=tension, 3=direct contradiction
# Rows = BSA, Columns = LGPD
conflict_matrix = np.array([
    [2, 0, 0, 0, 3],   # CIP collection
    [0, 3, 0, 0, 2],   # SAR tipping-off
    [1, 0, 2, 0, 2],   # 5-year retention
    [1, 0, 0, 3, 2],   # Cross-border transfer
    [3, 0, 1, 1, 2],   # EDD
])

fig, ax = plt.subplots(figsize=(12, 6.5))
colors_list = ['#f0f4f8', '#fef3c7', '#fbbf24', '#ef4444']
cmap = mcolors.LinearSegmentedColormap.from_list('conflict', colors_list, N=256)
im = ax.imshow(conflict_matrix, cmap=cmap, vmin=0, vmax=3, aspect='auto')

ax.set_xticks(range(len(lgpd_provisions)))
ax.set_yticks(range(len(bsa_obligations)))
ax.set_xticklabels(lgpd_provisions, fontsize=9.5)
ax.set_yticklabels(bsa_obligations, fontsize=9.5)

severity_labels = {0: '', 1: 'Ambiguity', 2: 'Tension', 3: 'Direct
Conflict'}
for i in range(len(bsa_obligations)):
    for j in range(len(lgpd_provisions)):
        val = conflict_matrix[i, j]
        label = severity_labels[val]
        if label:
            text_color = 'white' if val == 3 else 'black'
            ax.text(j, i, label, ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color=text_color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, aspect=15, pad=0.02)
cbar.set_ticks([0.375, 1.125, 1.875, 2.625])
cbar.set_ticklabels(['No conflict', 'Ambiguity', 'Regulatory tension', 'Direct contradiction'])
cbar.ax.tick_params(labelsize=9)

ax.set_title('BSA vs LGPD Obligation Conflict Matrix
'
             'For SEC-Registered Investment Advisers with Brazilian Client Exposure',
             fontsize=12, fontweight='bold', pad=14)
ax.set_xlabel('LGPD Provision', fontsize=10, labelpad=8)
ax.set_ylabel('BSA/AML Obligation', fontsize=10, labelpad=8)

plt.xticks(rotation=0, ha='center')
fig.text(0.5, -0.04,
    'Source: Author's analysis based on BSA (31 U.S.C. §§ 5311–5336), '
    '89 Fed. Reg. 72,156 (2024), LGPD Law 13,709/2018, '
    'ANPD Resolution No. 19/2024. See companion paper Section 4.',
    ha='center', fontsize=8, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('figure5_compliance_gap_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("Figure 5 saved.")
print("\nCritical conflicts (score=3):")
print("  CIP collection ↔ Legal Obligation Basis: unresolved scope")
print("  SAR tipping-off ↔ LGPD Transparency: directly contradictory")
print("  Cross-border transfer ↔ Art. 33 restrictions: no US adequacy decision")
print("  EDD ↔ Data Minimization: no ANPD sector guidance for cross-border advisers")


### Figure 6 — Regulatory Exposure Scoring Model

This scoring model estimates the probability of simultaneous BSA-LGPD exposure for a hypothetical investment adviser population, using observable proxies from public data. The model is not a compliance tool — it is an analytical instrument for prioritizing the policy guidance most urgently needed.

**Methodology:** Rule-based weighted scoring using five observable dimensions. Each dimension is scored 0–10 and weighted by its regulatory materiality. The output is a composite exposure score 0–100.


In [ ]:
np.random.seed(42)
n = 500  # Hypothetical adviser population

# ── Generate synthetic population ─────────────────────────────────────────────
# Based on observable Form ADV distributions
data = pd.DataFrame({
    'adviser_id': [f'RIA_{i:04d}' for i in range(n)],
    'raum_billion': np.random.lognormal(mean=2.5, sigma=1.8, size=n).clip(0.01, 5000),
    'pct_non_us_clients': np.random.beta(1.5, 6, size=n) * 100,
    'florida_hq': np.random.choice([0, 1], n, p=[0.87, 0.13]),
    'has_foreign_inst_clients': np.random.choice([0, 1], n, p=[0.95, 0.05]),
    'sar_filings_prior_year': np.random.poisson(lam=3, size=n),
})

# ── Score each dimension (0–10) ───────────────────────────────────────────────
data['score_raum'] = (np.log1p(data['raum_billion']) / np.log1p(5000) * 10).clip(0, 10)
data['score_non_us'] = (data['pct_non_us_clients'] / 100 * 10).clip(0, 10)
data['score_geography'] = data['florida_hq'] * 8 + (1 - data['florida_hq']) * 2
data['score_foreign_clients'] = data['has_foreign_inst_clients'] * 9
data['score_sar_history'] = (np.log1p(data['sar_filings_prior_year']) /
                               np.log1p(50) * 10).clip(0, 10)

# ── Weighted composite score ──────────────────────────────────────────────────
weights = {
    'score_raum': 0.20,
    'score_non_us': 0.30,
    'score_geography': 0.20,
    'score_foreign_clients': 0.20,
    'score_sar_history': 0.10,
}
data['exposure_score'] = sum(data[col] * w for col, w in weights.items())
data['risk_tier'] = pd.cut(data['exposure_score'],
                             bins=[0, 3, 5, 7, 10],
                             labels=['Low', 'Moderate', 'High', 'Critical'])

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Distribution
ax1 = axes[0]
tier_colors = {'Low': '#93c5fd', 'Moderate': '#fbbf24', 'High': '#f97316', 'Critical': '#ef4444'}
for tier, color in tier_colors.items():
    subset = data[data['risk_tier'] == tier]['exposure_score']
    ax1.hist(subset, bins=20, color=color, alpha=0.8, edgecolor='white',
             label=f'{tier} (n={len(subset)})')

ax1.set_title('BSA-LGPD Exposure Score Distribution
(Synthetic RIA Population, n=500)', pad=10)
ax1.set_xlabel('Composite Exposure Score (0–10)')
ax1.set_ylabel('Number of Advisers')
ax1.legend(title='Risk Tier', fontsize=9)
ax1.axvline(x=5, color=GRAY, linestyle='--', linewidth=1.2, alpha=0.7)
ax1.text(5.15, ax1.get_ylim()[1]*0.92, 'Regulatory priority\nthreshold', fontsize=8.5, color=GRAY)

# Right: Feature importance (correlation with score)
ax2 = axes[1]
features = ['Non-US clients %', 'Geography (FL)', 'Foreign Institution clients',
            'RAUM size', 'SAR filing history']
importances = [weights['score_non_us'], weights['score_geography'],
               weights['score_foreign_clients'], weights['score_raum'],
               weights['score_sar_history']]
colors_imp = [TEAL, GOLD, RED, NAVY, GRAY]

bars = ax2.barh(features, importances, color=colors_imp, edgecolor='white',
                linewidth=0.5, alpha=0.88)
ax2.set_title('Model Feature Weights
(BSA-LGPD Exposure Scoring)', pad=10)
ax2.set_xlabel('Weight in Composite Score')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
for bar, val in zip(bars, importances):
    ax2.text(val + 0.003, bar.get_y() + bar.get_height()/2,
             f'{val:.0%}', va='center', fontsize=10, fontweight='bold')

fig.text(0.5, -0.03,
    'Note: Scoring model is an analytical instrument for policy prioritization, not a compliance tool. '
    'Population is synthetic; distributions calibrated to Form ADV aggregate data (SEC, 2023).',
    ha='center', fontsize=8, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('figure6_exposure_scoring_model.png', bbox_inches='tight', dpi=150)
plt.show()

# Summary statistics
tier_counts = data['risk_tier'].value_counts()
print("Risk tier distribution:")
for tier in ['Low', 'Moderate', 'High', 'Critical']:
    pct = tier_counts.get(tier, 0) / n * 100
    print(f"  {tier}: {tier_counts.get(tier, 0):3d} advisers ({pct:.0f}%)")
print(f"\nMedian exposure score: {data['exposure_score'].median():.2f}")
print(f"Mean exposure score:   {data['exposure_score'].mean():.2f}")


## 5. Summary Statistics and Policy Implications

In [ ]:
print("=" * 68)
print("SUMMARY: KEY FINDINGS FOR COMPANION PAPER (LGPD-BSA, SSRN 2026)")
print("=" * 68)
print()
print("FINDING 1 — Investment adviser sector materiality:")
print(f"  IA SAR filings grew +1,009% from 2014 ({1638:,}) to 2023 ({18170:,})")
print(f"  IA share of Securities/Futures SARs: 7.4% (2014) → 29.8% (2023)")
print(f"  2024 drop of 34.3% (to {11935:,}) coincides with FinCEN rule delay")
print()
print("FINDING 2 — Geographic concentration of Brazil-facing exposure:")
print(f"  Florida securities SARs: +288% growth 2019→2024")
print(f"  Florida 2021 spike: +227% YoY (peak: {18146:,})")
print(f"  Florida 2024: {21510:,} (2nd highest on record)")
print()
print("FINDING 3 — Scale of LGPD-exposed data processing:")
print(f"  Non-US RAUM (2023): $30.3 trillion (23.5% of total RIA RAUM)")
print(f"  This represents the data of non-US persons processed by U.S.-regulated")
print(f"  entities — the direct subject of Art. 33 LGPD international transfer rules")
print()
print("FINDING 4 — Compliance urgency window:")
print(f"  FinCEN rule published: September 4, 2024 (89 Fed. Reg. 72,156)")
print(f"  FinCEN compliance deadline: January 1, 2028")
print(f"  ANPD SCCs mandatory: August 23, 2025 (Resolution 19/2024)")
print(f"  ANPD enforcement priority: Financial sector (Map 2026–2027)")
print()
print("FINDING 5 — Regulatory gap:")
print(f"  4 direct BSA-LGPD conflict categories identified (Section 4, companion paper)")
print(f"  Critical conflicts: SAR tipping-off ↔ LGPD transparency (Art. 6 VI)")
print(f"  Critical conflicts: Cross-border SAR transfer ↔ Art. 33 (no US adequacy)")
print(f"  Unresolved: ANPD has issued no guidance on foreign legal obligations")
print(f"              as retention basis under LGPD Art. 16 (as of March 2026)")
print()
print("=" * 68)
print("DATA SOURCES:")
print("  FinCEN SAR Filing Trend Data (2014–2024) — fincen.gov/reports/sar-stats")
print("  SEC IA Statistics, Form ADV Dec 2023 — sec.gov/files/im-investment-adviser-statistics-20240515.pdf")
print("  Companion paper: SSRN (2026) — LGPD Compliance for U.S. Investment Advisers")
print("  Author ORCID: 0009-0004-6401-3465")
print("=" * 68)


## 6. How to Cite This Work

```
Santos, W.M. (2026). AML/BSA Compliance Gaps in the Brazil–U.S. Financial Corridor:
Quantifying Regulatory Exposure for SEC-Registered Investment Advisers.
Kaggle Dataset / GitHub Repository. ORCID: 0009-0004-6401-3465.

Companion paper:
Santos, W.M. (2026). LGPD Compliance for U.S. Investment Advisers with Brazilian
Client Exposure: Navigating the Regulatory Gap Between the Bank Secrecy Act and
Brazil's Data Protection Framework. SSRN Working Paper.
```

## 7. Reproducibility Notes

All data used in this analysis is publicly available:
- **FinCEN SAR Statistics**: `https://www.fincen.gov/reports/sar-stats/sar-filings-industry`
- **SEC IA Statistics**: `https://www.sec.gov/data-research/statistics-data-visualizations/investment-adviser-statistics`

No confidential, proprietary, or personally identifiable data was used.
All figures generated from this notebook are released under CC BY 4.0.
